# Check de la key de Deep Seek

Test rápido para verificar que tu `DEEPSEEK_API_KEY` es válida y funciona

In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    print("DEEPSEEK_API_KEY not found. Make sure it's set in your .env file.")
else:
    print(f"Key found: {api_key[:8]}...{api_key[-4:]}")


MODEL_NAME = "deepseek-v4-flash"
#MODEL_NAME = "deepseek-v4-pro"

Key found: sk-324fa...fe55


In [ ]:
from openai import OpenAI


client = OpenAI(    
    api_key=api_key,
    base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello"},
    ],
    stream=False,
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}}
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


# Ejemplo básico workflow con Agentes

In [11]:
def agent_developer(prompt: str, review_feedback: str = None) -> str:
    """Agent 1: Writes initial code OR refines code based on feedback."""
    
    system_prompt = (
        "You are an expert Software Engineer. Write clean, well-documented, "
        "and efficient Python code. Return ONLY valid Python code inside markdown blocks."
    )
    
    # If there is feedback, this is a refinement task (Round 2)
    if review_feedback:
        print(f"\n[Agent: Developer] Refactoring code based on peer review feedback...")
        user_content = (
            f"Please refactor your previous implementation based on this review code feedback:\n\n"
            f"--- REVIEW FEEDBACK ---\n{review_feedback}\n\n"
            f"Provide the updated, secure, and corrected Python code."
        )
    # If no feedback, this is the initial build (Round 1)
    else:
        print(f"\n[Agent: Developer] Writing initial code implementation...")
        user_content = f"Write a Python function for the following requirement: {prompt}"

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ],
        temperature=0.1 # Low temperature for consistent syntax
    )
    return response.choices[0].message.content


def agent_code_reviewer(draft_code: str) -> str:
    """Agent 2: Acts as a Senior Tech Lead reviewing the code for vulnerabilities and edge cases."""
    print(f"\n[Agent: Reviewer] Analyzing code for bugs, security issues, and edge cases...")
    
    system_prompt = (
        "You are a Senior Technical Lead performing a strict code review. "
        "Analyze the provided code specifically looking for:\n"
        "1. Security vulnerabilities (e.g., injection, unsafe parsing)\n"
        "2. Edge cases (e.g., handling None/Null values, empty inputs, division by zero)\n"
        "3. Performance optimizations\n"
        "Provide a clear, bulleted list of issues found. Do not write the code yourself."
    )
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Review this code snippet:\n{draft_code}"}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

In [ ]:
dev_requirement = (
        "Create a function called 'evaluate_expression' that takes a string mathematical expression "
        "from a user input (like '2 + 3') and evaluates it to return the numeric result."
    )
    

# Round 1: Developer builds initial code
initial_code = agent_developer(prompt=dev_requirement)
review_notes = agent_code_reviewer(draft_code=initial_code)
final_code = agent_developer(prompt=dev_requirement, review_feedback=review_notes)


[Agent: Developer] Writing initial code implementation...

>>> INITIAL CODE DRAFT <<<
```python
import ast
import operator
from typing import Union, Any

# Mapping from AST operator nodes to Python operator functions
_BIN_OP_MAP = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}

_UNARY_OP_MAP = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}

def _safe_eval_node(node: ast.AST) -> Union[int, float]:
    """
    Recursively evaluate an AST node containing only numbers and operators.
    Raises ValueError for unsupported node types or invalid operations.
    """
    # Literal numbers (Python 3.7 and below use ast.Num; 3.8+ uses ast.Constant)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    # Handle ast.Num for older Python versions (optional)
    if isinstanc

In [12]:
from IPython.display import Markdown, display
print("\n--- Initial Code ---")
display(Markdown(initial_code))
print("\n--- Review Feedback ---")
display(Markdown(review_notes))
print("\n--- Final Code After Refinement ---")
display(Markdown(final_code))


--- Initial Code ---


```python
import ast
import operator
from typing import Union, Any

# Mapping from AST operator nodes to Python operator functions
_BIN_OP_MAP = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}

_UNARY_OP_MAP = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}

def _safe_eval_node(node: ast.AST) -> Union[int, float]:
    """
    Recursively evaluate an AST node containing only numbers and operators.
    Raises ValueError for unsupported node types or invalid operations.
    """
    # Literal numbers (Python 3.7 and below use ast.Num; 3.8+ uses ast.Constant)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    # Handle ast.Num for older Python versions (optional)
    if isinstance(node, ast.Num):
        return node.n

    # Binary operation (e.g., a + b)
    if isinstance(node, ast.BinOp):
        left = _safe_eval_node(node.left)
        right = _safe_eval_node(node.right)
        op_func = _BIN_OP_MAP.get(type(node.op))
        if op_func is None:
            raise ValueError(f"Unsupported binary operator: {type(node.op).__name__}")
        try:
            return op_func(left, right)
        except ZeroDivisionError:
            raise ValueError("Division by zero is not allowed")

    # Unary operation (e.g., -a or +a)
    if isinstance(node, ast.UnaryOp):
        operand = _safe_eval_node(node.operand)
        op_func = _UNARY_OP_MAP.get(type(node.op))
        if op_func is None:
            raise ValueError(f"Unsupported unary operator: {type(node.op).__name__}")
        return op_func(operand)

    # If we get here, the node type is not allowed
    raise ValueError(f"Unsupported syntax: {type(node).__name__}")


def evaluate_expression(expression: str) -> Union[int, float]:
    """
    Safely evaluate a simple arithmetic expression provided as a string.

    Supports: +, -, *, /, //, %, **, parentheses, unary + and -.
    Does not allow function calls, variables, or any other Python constructs.

    Parameters
    ----------
    expression : str
        A mathematical expression (e.g., '2 + 3 * (4 - 1)')

    Returns
    -------
    int or float
        The numeric result of evaluating the expression.

    Raises
    ------
    ValueError
        If the expression is syntactically invalid or contains unsupported operations.

    Examples
    --------
    >>> evaluate_expression('2 + 3')
    5
    >>> evaluate_expression('10 / 3')
    3.3333333333333335
    >>> evaluate_expression('2 ** 3')
    8
    """
    try:
        # Parse the expression into an AST
        tree = ast.parse(expression.strip(), mode='eval')

        # Evaluate the AST recursively
        result = _safe_eval_node(tree.body)
        return result
    except SyntaxError as e:
        raise ValueError(f"Invalid expression syntax: {e}")
    except Exception as e:
        raise ValueError(f"Evaluation failed: {e}")


# Example usage (runs only when script is executed directly)
if __name__ == "__main__":
    test_expressions = [
        "2 + 3",
        "10 - 2 * 3",
        "(5 + 3) * 2",
        "100 / 4",
        "2 ** 3",
        "7 // 2",
        "10 % 3",
        "-5 + 3",
        "+2 + (+3)",
    ]
    for expr in test_expressions:
        try:
            result = evaluate_expression(expr)
            print(f"{expr} = {result}")
        except ValueError as err:
            print(f"{expr} -> Error: {err}")
```


--- Review Feedback ---


### Security Vulnerabilities

- **Stack overflow via deeply nested expressions**: The recursive `_safe_eval_node` can exhaust the Python call stack for expressions with extreme nesting (e.g., `(((...)))`). An attacker can craft a small payload that causes `RecursionError`, crashing the program. Consider using an iterative evaluation or limiting recursion depth.

- **Boolean values accepted as numeric**: `bool` is a subclass of `int`, so `True` and `False` are allowed as constants (e.g., `True + 2` returns `3`). This violates the documented restriction to “only numbers” and may enable unintended behavior. Explicitly exclude booleans: `isinstance(node.value, (int, float)) and not isinstance(node.value, bool)`.

- **No input length or complexity limits**: Extremely long or purposely crafted expressions can consume excessive memory or CPU during `ast.parse` or evaluation (e.g., `2**999999`, deeply nested operations). This could be used for denial-of-service. Add a maximum string length and optionally a recursion depth cap.

- **Broad exception catch hides `RecursionError` and `MemoryError`**: The `except Exception` in `evaluate_expression` catches and converts these to `ValueError`, suppressing diagnostic information. Consider catching only expected exceptions (`SyntaxError`, `ValueError`, `ZeroDivisionError`), and letting fatal errors propagate.

### Edge Cases

- **Empty or whitespace-only input**: `expression.strip()` yields `""`, which causes `ast.parse` to raise `SyntaxError`. The error message becomes `"Invalid expression syntax: unexpected EOF while parsing"`, which is unclear. Provide a specific check: if `expression.strip()` is empty, raise `ValueError("Empty expression")`.

- **Non-string input**: If `expression` is `None`, `bytes`, or another type, `expression.strip()` raises `AttributeError`. This is caught by the generic `Exception` handler, but the error message is unhelpful. Add an early type check: `if not isinstance(expression, str): raise ValueError("Input must be a string")`.

- **Division by zero in `%` (modulo)**: Though not tested in the example, `operator.mod` also raises `ZeroDivisionError`. The existing catch handles it correctly.

- **Large exponentiation**: `2 ** 1000000` can consume huge memory and time, potentially freezing the application. No guard against this. Consider limiting exponent magnitude or using a timeout.

- **Float vs int type ambiguity**: The return type annotation `Union[int, float]` is correct, but operations like `//` on floats return float, not int. This is Python’s behavior and not a bug, but users may expect `int` for floor division of integers. Documentation should be clear.

### Performance Optimizations

- **Recursive evaluation for deeply nested expressions**: The recursion depth of `_safe_eval_node` is equal to the nesting depth of the AST. For deeply nested expressions (e.g., a long chain of `+`), this can hit Python’s recursion limit (~1000) or cause stack overhead. Consider implementing an iterative, stack-based evaluation to avoid recursion entirely.

- **Repeated `ast.parse` for same expression**: No caching is used. If the same expression is evaluated many times, parsing it each time is wasteful. For repeated use, consider caching the compiled AST.

- **Broad exception handling inside loop**: In `_safe_eval_node`, each `BinOp` and `UnaryOp` recalculates the operator function from the map. This is acceptable, but micro‑optimizations (e.g., using `operator` directly) are already in place. No major performance gain needed for typical usage.


--- Final Code After Refinement ---


```python
import ast
import operator
from functools import lru_cache
from typing import Union

# Constants for security limits
MAX_EXPRESSION_LENGTH = 2000
MAX_DEPTH = 100
MAX_EXPONENT = 1000
MAX_RESULT_ABS = 1e100

# Allowed operators mapping
ALLOWED_BIN_OP = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}

ALLOWED_UNARY_OP = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}


class SafeEvaluator:
    """
    Safely evaluate arithmetic expressions containing only numbers, parentheses,
    and the operators: +, -, *, /, //, %, **.

    The evaluator is designed to be resistant to:
    - Stack overflow (iterative evaluation with depth limit)
    - Denial of service (input length and exponent limits)
    - Unallowed node types and values (booleans, strings, etc.)
    - RecursionError and MemoryError (specific exception handling)
    """

    @lru_cache(maxsize=128)
    def evaluate(self, expression: str) -> Union[int, float]:
        """
        Evaluate a mathematical expression string and return the numeric result.

        Args:
            expression: A string containing a valid arithmetic expression.

        Returns:
            The evaluated numeric result (int or float).

        Raises:
            ValueError: If the expression is invalid, empty, too long, contains
                        unsupported syntax, or causes a calculation error.
            TypeError: If the input is not a string.
        """
        # Validate input type
        if not isinstance(expression, str):
            raise TypeError("Input must be a string")

        # Strip whitespace and check emptiness
        stripped = expression.strip()
        if not stripped:
            raise ValueError("Empty expression")

        # Enforce maximum length
        if len(stripped) > MAX_EXPRESSION_LENGTH:
            raise ValueError("Expression too long")

        try:
            tree = ast.parse(stripped, mode='eval')
        except SyntaxError as e:
            raise ValueError(f"Invalid expression syntax: {e}")

        # The parsed tree is an ast.Expression node; its body is the actual expression
        result = self._evaluate_node(tree.body)
        return result

    def _evaluate_node(self, node: ast.AST) -> Union[int, float]:
        """
        Iteratively evaluate an AST node representing an arithmetic expression.

        Uses a stack for post-order traversal to avoid recursion depth issues.
        """
        # Stack elements: (node, state, depth)
        # state = 0: children not processed yet; state = 1: children processed
        stack = [(node, 0, 0)]
        # Stack for intermediate results (operands)
        results = []

        while stack:
            current, state, depth = stack.pop()

            if state == 0:
                # --- First encounter: validate and push children ---
                if depth > MAX_DEPTH:
                    raise ValueError("Expression too deeply nested")

                if isinstance(current, ast.Constant):
                    # Ensure it's a numeric constant (and not a boolean)
                    if isinstance(current.value, bool) or not isinstance(current.value, (int, float)):
                        raise ValueError("Only numeric constants are allowed")
                    # Push constant with state=1 so it will be handled in the next pop
                    stack.append((current, 1, depth))
                    # Also push its value onto results (they will be used when parent processes)
                    results.append(current.value)

                elif isinstance(current, ast.BinOp):
                    # Validate operator
                    op_type = type(current.op)
                    if op_type not in ALLOWED_BIN_OP:
                        raise ValueError(f"Operator not allowed: {op_type.__name__}")
                    # Push the node again with state=1 for final evaluation
                    stack.append((current, 1, depth))
                    # Push right operand first (so left is on top after popping)
                    # We use a helper to correctly set depth for children
                    stack.append((current.right, 0, depth + 1))
                    stack.append((current.left, 0, depth + 1))

                elif isinstance(current, ast.UnaryOp):
                    op_type = type(current.op)
                    if op_type not in ALLOWED_UNARY_OP:
                        raise ValueError(f"Unary operator not allowed: {op_type.__name__}")
                    stack.append((current, 1, depth))
                    stack.append((current.operand, 0, depth + 1))

                elif isinstance(current, ast.Expression):
                    # Unwrap the expression; this node is transparent
                    stack.append((current.body, 0, depth))

                else:
                    raise ValueError("Unsupported syntax in expression")

            else:
                # state == 1: children have been processed, now compute
                if isinstance(current, ast.Constant):
                    # Results already on stack, nothing to do
                    continue

                elif isinstance(current, ast.BinOp):
                    # Pop operands: right was pushed after left, so left is on top
                    # Actually our push order put right first, then left. So when popping:
                    # The last pushed is left (since we push right then left, left is top)
                    # Let's verify: push right, push left -> stack top = left. So pop left first.
                    # But we need left then right for binary operator. So we pop left and right.
                    right = results.pop()
                    left = results.pop()
                    # Correct order: left op right
                    op_func = ALLOWED_BIN_OP[type(current.op)]
                    try:
                        result = op_func(left, right)
                    except ZeroDivisionError:
                        raise ValueError("Division by zero")
                    except OverflowError:
                        raise ValueError("Result too large (overflow)")

                    # Additional safety: limit exponent result magnitude
                    if abs(result) > MAX_RESULT_ABS:
                        raise ValueError("Result too large")

                    results.append(result)

                elif isinstance(current, ast.UnaryOp):
                    operand = results.pop()
                    op_func = ALLOWED_UNARY_OP[type(current.op)]
                    result = op_func(operand)
                    if abs(result) > MAX_RESULT_ABS:
                        raise ValueError("Result too large")
                    results.append(result)

        if len(results) != 1:
            raise ValueError("Internal evaluation error")
        return results[0]


# Convenience function (optional)
def safe_eval(expression: str) -> Union[int, float]:
    """
    Evaluate a mathematical expression safely.

    This is a wrapper around SafeEvaluator for quick use.
    """
    evaluator = SafeEvaluator()
    return evaluator.evaluate(expression)
```

# Ejemplo: Delegación entre Agentes (Agent Tool Pattern)

Traducción del patrón de Google ADK donde un agente padre delega a un sub-agente que tiene herramientas.

**Flujo:**
1. `artist_agent` inventa un prompt creativo para una imagen
2. Llama a `generate_image_via_subagent` que delega al sub-agente `ImageGen`
3. `ImageGen` usa la tool `generate_image` para generar la imagen
4. El resultado sube de vuelta al agente padre

In [ ]:
import json

# 1. Tool function — the core capability separated from reasoning
def generate_image(prompt: str) -> dict:
    """Generates an image based on a textual prompt."""
    print(f"TOOL: Generating image for prompt: '{prompt}'")
    mock_image_bytes = "bW9ja19pbWFnZV9kYXRhX2Zvcl9hX2NhdF93ZWFyaW5nX2FfaGF0"  # base64 mock
    return {
        "status": "success",
        "image_base64": mock_image_bytes,
        "mime_type": "image/png",
    }


# 2. Tool schema for OpenAI function calling
generate_image_tool = {
    "type": "function",
    "function": {
        "name": "generate_image",
        "description": "Generates an image based on a detailed text prompt.",
        "parameters": {
            "type": "object",
            "properties": {
                "prompt": {
                    "type": "string",
                    "description": "A detailed description of the image to generate.",
                }
            },
            "required": ["prompt"],
        },
    },
}


# 3. Sub-agent: ImageGen — equivalent to the Google ADK LlmAgent
def image_generator_agent(user_prompt: str) -> dict:
    """Sub-agent that uses the generate_image tool to create images."""
    print(f"\n[Agent: ImageGen] Received request: '{user_prompt}'")

    messages = [
        {
            "role": "system",
            "content": (
                "You are an image generation specialist. Use the `generate_image` tool "
                "to create the image described by the user. Pass the user's full request "
                "as the 'prompt' argument."
            ),
        },
        {"role": "user", "content": user_prompt},
    ]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=[generate_image_tool],
        tool_choice="auto",
    )

    msg = response.choices[0].message

    # If the model called the tool, execute it and return the result
    if msg.tool_calls:
        tool_call = msg.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        result = generate_image(**args)
        print(f"[Agent: ImageGen] Tool returned status: {result['status']}")
        return result

    # Fallback: model responded with text instead of a tool call
    print(f"[Agent: ImageGen] No tool call, text response: {msg.content}")
    return {"status": "error", "message": msg.content}


# 4. Wrapper tool schema — this is what the parent agent sees (equivalent to AgentTool)
image_subagent_tool = {
    "type": "function",
    "function": {
        "name": "generate_image_via_subagent",
        "description": "Use this tool to generate an image. The input should be a descriptive prompt of the desired image.",
        "parameters": {
            "type": "object",
            "properties": {
                "prompt": {
                    "type": "string",
                    "description": "A descriptive prompt of the desired image.",
                }
            },
            "required": ["prompt"],
        },
    },
}


def generate_image_via_subagent(prompt: str) -> dict:
    """Delegates to the ImageGen sub-agent."""
    return image_generator_agent(prompt)


# Registry to dispatch tool calls by name
TOOL_DISPATCH = {
    "generate_image_via_subagent": generate_image_via_subagent,
}

print("Tools and agents defined.")

In [ ]:
# 5. Parent agent: Artist — invents a creative prompt and delegates to ImageGen

def artist_agent():
    """Parent agent that creates a prompt and delegates image generation."""
    print("[Agent: Artist] Starting creative process...\n")

    messages = [
        {
            "role": "system",
            "content": (
                "You are a creative artist. First, invent a creative and descriptive prompt "
                "for an image. Then, use the `generate_image_via_subagent` tool to generate "
                "the image using your prompt."
            ),
        },
        {"role": "user", "content": "Create an interesting and unique piece of art."},
    ]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=[image_subagent_tool],
        tool_choice="auto",
    )

    msg = response.choices[0].message

    if msg.tool_calls:
        tool_call = msg.tool_calls[0]
        fn_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f"[Agent: Artist] Calling tool '{fn_name}' with prompt: '{args['prompt'][:80]}...'\n")

        result = TOOL_DISPATCH[fn_name](**args)

        # Send tool result back to the parent agent for final response
        messages.append(msg.model_dump())
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result),
        })

        final = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
        )
        print(f"\n[Agent: Artist] Final response:\n{final.choices[0].message.content}")
        return result

    print(f"[Agent: Artist] No tool call: {msg.content}")
    return None


# Run the full pipeline
result = artist_agent()